# Chapter 18 — Gradient Descent in Practice

*From Absolute Zero* — companion notebook.

Every block below is the code printed in the chapter, in the same order. Run the cells top to bottom; the output should match the book exactly. If it does not, check `requirements.txt` first, then `docs/TROUBLESHOOTING.md`.

In [1]:
!pip -q install -r https://raw.githubusercontent.com/FromAbsoluteZero/CodeBase/main/requirements.txt  # Colab only; skip locally

zsh:1: command not found: pip


## Shared setup

Imports and the objects the blocks below reuse. The chapter prints these once and then continues the same session. This cell is `code/ch18/_lib.py`.

In [2]:
import numpy as np, warnings; warnings.filterwarnings("ignore")

def make_problem(cond=1.0, n=800, d=2, seed=0):
    """A least-squares problem whose curvature ratio we control."""
    rng = np.random.default_rng(seed)
    X = rng.normal(size=(n, d))
    X[:, 1] *= cond                       # stretch one axis
    w_true = np.array([2.0, -1.0])
    y = X @ w_true + rng.normal(0, 0.3, n)
    return X, y, w_true

def loss(X, y, w):
    r = X @ w - y
    return float(r @ r / len(y))

def grad(X, y, w):
    return 2 * X.T @ (X @ w - y) / len(y)

## The chapter code

### Block 1  (`c1.py`)

In [3]:
# The learning rate is the one setting that decides whether training
# works at all. Chapter 10 showed this on one problem; here is the rule.
X, y, w_true = make_problem()
L = 2 * np.linalg.eigvalsh(X.T @ X / len(y)).max()    # curvature bound
print(f"largest curvature of this loss: {L:.3f}")
print(f"gradient descent diverges above eta = 2/L = {2/L:.4f}\n")

print(f"{'eta':>8}{'loss after 200 steps':>24}{'verdict':>14}")
for eta in (0.001, 0.01, 0.1, 0.5, 0.9, 1.05):
    w = np.zeros(2)
    for _ in range(200):
        w -= eta * grad(X, y, w)
        if not np.isfinite(w).all():
            break
    l = loss(X, y, w)
    verdict = ("diverged" if not np.isfinite(l) or l > 1e3
               else "crawling" if l > 0.2 else "converged")
    shown = "overflow" if not np.isfinite(l) else f"{l:.4f}"
    print(f"{eta:>8.3f}{shown:>24}{verdict:>14}")

largest curvature of this loss: 1.998
gradient descent diverges above eta = 2/L = 1.0011

     eta    loss after 200 steps       verdict
   0.001                  2.3265      crawling
   0.010                  0.0967     converged
   0.100                  0.0951     converged
   0.500                  0.0951     converged
   0.900                  0.0951     converged
   1.050  72961621552780416.0000      diverged


### Block 2  (`c2.py`)

In [4]:
# Why plain descent is slow: it is not the learning rate, it is the shape.
print(f"{'stretch':>9}{'condition':>12}{'steps to loss<0.11':>21}"
      f"{'best eta':>10}")
for c in (1, 3, 10, 30):
    X, y, _ = make_problem(cond=c)
    H = 2 * X.T @ X / len(y)
    ev = np.linalg.eigvalsh(H)
    best, best_steps = None, None
    for eta in np.geomspace(1e-4, 2 / ev.max() * 0.99, 60):
        w, steps = np.zeros(2), None
        for i in range(1, 4001):
            w -= eta * grad(X, y, w)
            if not np.isfinite(w).all():
                break
            if loss(X, y, w) < 0.11:
                steps = i
                break
        if steps and (best_steps is None or steps < best_steps):
            best, best_steps = eta, steps
    print(f"{c:>9}{ev.max()/ev.min():>12.1f}{best_steps:>21}{best:>10.4f}")

  stretch   condition   steps to loss<0.11  best eta


        1         1.1                    2    0.3888


        3         8.9                   14    0.1007


       10        99.2                  150    0.0094


       30       892.7                 1256    0.0011


### Block 3  (`c3.py`)

In [5]:
# Momentum accumulates a velocity, so consistent directions build speed
# and oscillating ones cancel. Adam also rescales each coordinate.
X, y, _ = make_problem(cond=10)      # condition number about 99

def run(kind, eta, steps=400, beta=0.9, b1=0.9, b2=0.999, eps=1e-8):
    w, v, m, s = np.zeros(2), np.zeros(2), np.zeros(2), np.zeros(2)
    hist = []
    for t in range(1, steps + 1):
        g = grad(X, y, w)
        if kind == "plain":
            w -= eta * g
        elif kind == "momentum":
            v = beta * v + g
            w -= eta * v
        else:                                   # adam
            m = b1 * m + (1 - b1) * g
            s = b2 * s + (1 - b2) * g * g
            mh, sh = m / (1 - b1**t), s / (1 - b2**t)
            w -= eta * mh / (np.sqrt(sh) + eps)
        hist.append(loss(X, y, w))
    return hist

for kind, eta in [("plain", 0.0094), ("momentum", 0.0035), ("adam", 0.10)]:
    h = run(kind, eta)
    first = next((i for i, l in enumerate(h, 1) if l < 0.11), None)
    print(f"{kind:<10} eta {eta:<7} loss<0.11 at step "
          f"{first if first else '>400':<6} final {h[-1]:.4f}")
print("\nmomentum's effective step is larger than its learning rate,")
print("so it usually wants a smaller eta than plain descent.")

plain      eta 0.0094  loss<0.11 at step 151    final 0.0951
momentum   eta 0.0035  loss<0.11 at step 59     final 0.0951
adam       eta 0.1     loss<0.11 at step 50     final 0.0951

momentum's effective step is larger than its learning rate,
so it usually wants a smaller eta than plain descent.


### Block 4  (`c4.py`)

In [6]:
# Full-batch descent uses every row per step. Mini-batches use a sample,
# so each step is noisier and vastly cheaper.
rng = np.random.default_rng(0)
X, y, _ = make_problem(cond=10, n=8000)

def sgd(batch, eta, epochs=8):
    w = np.zeros(2)
    n = len(y)
    per_epoch = max(n // batch, 1)
    hist = []
    for _ in range(epochs):
        idx = rng.permutation(n)
        for b in range(per_epoch):
            sl = idx[b * batch:(b + 1) * batch]
            w -= eta * grad(X[sl], y[sl], w)
        hist.append(loss(X, y, w))
    return hist, per_epoch * epochs

print(f"{'batch size':>11}{'updates':>9}{'rows seen':>11}{'final loss':>12}")
for batch in (8000, 512, 64, 8):
    h, upd = sgd(batch, 0.0094 if batch == 8000 else 0.005)
    print(f"{batch:>11}{upd:>9}{upd*batch:>11,}{h[-1]:>12.4f}")

 batch size  updates  rows seen  final loss
       8000        8     64,000     12.7836
        512      120     61,440      0.4463
         64     1000     64,000      0.0922
          8     8000     64,000      0.0922


### Block 5  (`c5.py`)

In [7]:
# Schedules earn their place when the gradient is noisy. With mini-batches
# a constant rate plateaus at a noise floor it cannot get below.
X, y, _ = make_problem(cond=10, n=8000)
best = loss(X, y, np.linalg.lstsq(X, y, rcond=None)[0])
ETA0, BATCH, EPOCHS = 0.006, 64, 12

def run(sched):
    rng = np.random.default_rng(0)
    w = np.zeros(2)
    per_epoch = len(y) // BATCH
    total = per_epoch * EPOCHS
    t = 0
    for _ in range(EPOCHS):
        idx = rng.permutation(len(y))
        for b in range(per_epoch):
            t += 1
            if sched == "constant":
                eta = ETA0
            elif sched == "step":
                eta = ETA0 * (0.1 ** (t // (total // 3)))
            else:                                # cosine decay to zero
                eta = ETA0 * 0.5 * (1 + np.cos(np.pi * t / total))
            sl = idx[b * BATCH:(b + 1) * BATCH]
            w -= eta * grad(X[sl], y[sl], w)
    return loss(X, y, w)

for s in ("constant", "step", "cosine"):
    l = run(s)
    print(f"{s:<10} final loss {l:.6f}   excess over best {l - best:.2e}")
print(f"\nbest achievable on this data: {best:.6f}")

constant   final loss 0.088953   excess over best 1.00e-04
step       final loss 0.088865   excess over best 1.25e-05
cosine     final loss 0.088857   excess over best 4.87e-06

best achievable on this data: 0.088853
